In [ ]:
# install.packages("readxl")

In [ ]:
# Install only once
# install.packages(c("lavaan", "readxl", "modsem"))


In [ ]:
library(lavaan)
library(readxl)
library(modsem)
library(dplyr)
library(semTools)

In [ ]:

# ================================================================
# COMPLETE CB-SEM MODEL IN LAVAAN
# Measurement model + Main-effects structural model +
# Full structural model with four latent interactions
# Dataset: Alternate5_2.xlsx | N = 405
# ================================================================

# 1. Install and load required packages
# packages <- c("lavaan", "readxl", "dplyr", "semTools")

#new_packages <- packages[!(packages %in% installed.packages()[, "Package"])]
#if (length(new_packages) > 0) install.packages(new_packages)

library(lavaan)
library(readxl)
library(modsem)
library(dplyr)
library(semTools)

# 2. Import dataset
# 2. Import dataset
data <- read_excel("Data.xlsx", sheet = "Raw Data")
data <- as.data.frame(data)

# 3. Define measurement indicators
DIC  <- paste0("DIC", 1:4)
DIQ  <- paste0("DIQ", 1:5)
RI   <- paste0("RI", 1:5)
RCC  <- paste0("RCC", 1:4)
SCR  <- paste0("SCR", 1:5)
SP   <- paste0("SP", 1:4)
SA   <- paste0("SA", 1:4)
WR   <- c("WR1", "WR3", "WR4", "WR5")  # WR2 removed
EU   <- paste0("EU", 1:5)
FSOC <- paste0("FSOC", 1:4)
PSVC <- paste0("PSVC", 1:4)

all_items <- c(DIC, DIQ, RI, RCC, SCR, SP, SA, WR, EU, FSOC, PSVC)

# Check that all required indicators exist
missing_items <- setdiff(all_items, names(data))
if (length(missing_items) > 0) {
  stop(paste("Missing indicators:", paste(missing_items, collapse = ", ")))
}

# Convert measurement indicators to numeric
data[all_items] <- lapply(data[all_items], function(x) as.numeric(as.character(x)))

# Check sample and missing values
cat("\n================ DATA CHECK ================\n")
cat("Number of observations:", nrow(data), "\n")
cat("Number of measurement indicators:", length(all_items), "\n")
cat("Missing values in measurement indicators:",
    sum(is.na(data[all_items])), "\n")

if (sum(is.na(data[all_items])) > 0) {
  cat("FIML will be used to handle missing observations.\n")
}

# 4. Measurement model
measurement_model <- '
DIC  =~ DIC1 + DIC2 + DIC3 + DIC4
DIQ  =~ DIQ1 + DIQ2 + DIQ3 + DIQ4 + DIQ5
RI   =~ RI1 + RI2 + RI3 + RI4 + RI5
RCC  =~ RCC1 + RCC2 + RCC3 + RCC4
SCR  =~ SCR1 + SCR2 + SCR3 + SCR4 + SCR5
SP   =~ SP1 + SP2 + SP3 + SP4
SA   =~ SA1 + SA2 + SA3 + SA4
WR   =~ WR1 + WR3 + WR4 + WR5
EU   =~ EU1 + EU2 + EU3 + EU4 + EU5
FSOC =~ FSOC1 + FSOC2 + FSOC3 + FSOC4
PSVC =~ PSVC1 + PSVC2 + PSVC3 + PSVC4
'

# 5. Estimate measurement model (CFA)
fit_cfa <- cfa(
  model = measurement_model,
  data = data,
  estimator = "ML",
  std.lv = TRUE,
  missing = "fiml"
)

cat("\n\n================================================\n")
cat("RESULT 1: MEASUREMENT MODEL (CFA)\n")
cat("================================================\n")

print(summary(fit_cfa, fit.measures = TRUE, standardized = TRUE))

cat("\nMEASUREMENT MODEL FIT INDICES\n")
print(fitMeasures(
  fit_cfa,
  c("chisq", "df", "pvalue", "chisq.scaled",
    "cfi", "tli", "rmsea", "rmsea.ci.lower",
    "rmsea.ci.upper", "srmr", "gfi", "agfi", "nfi")
))

cat("\nSTANDARDIZED FACTOR LOADINGS\n")
print(
  standardizedSolution(fit_cfa) %>%
    filter(op == "=~") %>%
    select(lhs, rhs, est.std, se, z, pvalue)
)

# 6. Main-effects structural model (without interactions)
# This model is used for the main structural model fit indices.

main_structural_model <- paste0(measurement_model, '

# Human capability pathway
RI  ~ H1*WR
RCC ~ H2*RI

# Digital capability pathway
DIQ ~ H3*DIC
RCC ~ H4*DIQ

# Strategic capability pathway
SA  ~ H5*SP
RCC ~ H6*SA

# Resilience pathway
SCR ~ H7*RCC + EU_direct*EU

# Downstream outcomes
FSOC ~ H15*SCR
PSVC ~ H16*FSOC + H17*SCR

# Indirect effects
H8  := H1*H2
H9  := H3*H4
H10 := H5*H6
H16_indirect := H15*H16
H17_indirect_via_FSOC := H15*H16
H_RCC_to_PSVC_via_SCR := H7*H17
H_RCC_to_PSVC_serial := H7*H15*H16
')

fit_main <- sem(
  model = main_structural_model,
  data = data,
  estimator = "ML",
  std.lv = TRUE,
  missing = "fiml"
)


cat("\n\n================================================\n")
cat("RESULT 2: STRUCTURAL MODEL WITHOUT INTERACTIONS\n")
cat("================================================\n")

print(summary(fit_main, fit.measures = TRUE, standardized = TRUE, rsquare = TRUE))

cat("\nMAIN-EFFECTS STRUCTURAL MODEL FIT INDICES\n")
print(fitMeasures(
  fit_main,
  c("chisq", "df", "pvalue", "cfi", "tli",
    "rmsea", "rmsea.ci.lower", "rmsea.ci.upper",
    "srmr", "gfi", "agfi", "nfi")
))

cat("\nMAIN-EFFECTS STRUCTURAL PATHS\n")
print(
  standardizedSolution(fit_main) %>%
    filter(op == "~") %>%
    select(lhs, rhs, est.std, se, z, pvalue)
)

cat("\nR-SQUARED VALUES: MAIN-EFFECTS MODEL\n")
print(inspect(fit_main, "r2"))

# 7. Prepare product indicators for latent interactions
# Mean-center the original indicators and create all pairwise products.
# Product indicators are then mean-centered (double mean centering).

interaction_data <- data

interaction_pairs <- list(
  RI_DIQ = list(x = RI,   z = DIQ),
  DIQ_SA = list(x = DIQ,  z = SA),
  RI_SA  = list(x = RI,   z = SA),
  RCC_EU = list(x = RCC,  z = EU)
)

# Mean-center indicators used in interactions
interaction_items <- unique(unlist(
  lapply(interaction_pairs, function(x) c(x$x, x$z))
))

for (v in interaction_items) {
  interaction_data[[v]] <- interaction_data[[v]] -
    mean(interaction_data[[v]], na.rm = TRUE)
}

# Generate all pairwise product indicators
interaction_indicator_names <- list()

for (interaction_name in names(interaction_pairs)) {

  x_items <- interaction_pairs[[interaction_name]]$x
  z_items <- interaction_pairs[[interaction_name]]$z

  product_names <- character(0)

  for (x_item in x_items) {
    for (z_item in z_items) {

      product_name <- paste0(x_item, "_", z_item)

      product_value <- interaction_data[[x_item]] *
        interaction_data[[z_item]]

      # Double mean centering of product indicators
      product_value <- product_value -
        mean(product_value, na.rm = TRUE)

      interaction_data[[product_name]] <- product_value
      product_names <- c(product_names, product_name)
    }
  }

  interaction_indicator_names[[interaction_name]] <- product_names
}

# 8. Build interaction measurement model automatically
interaction_measurement_model <- paste0(
  "RIxDIQ =~ ",
  paste(interaction_indicator_names$RI_DIQ, collapse = " + "),
  "\n",
  "DIQxSA =~ ",
  paste(interaction_indicator_names$DIQ_SA, collapse = " + "),
  "\n",
  "RIxSA =~ ",
  paste(interaction_indicator_names$RI_SA, collapse = " + "),
  "\n",
  "RCCxEU =~ ",
  paste(interaction_indicator_names$RCC_EU, collapse = " + "),
  "\n"
)

# 9. Full structural model with interaction terms
# The measurement model remains the same.
# Four interaction factors are added to the structural model.

full_interaction_model <- paste0(
  measurement_model,
  "\n",
  interaction_measurement_model,
  '

# Human capability pathway
RI  ~ H1*WR
RCC ~ H2*RI

# Digital capability pathway
DIQ ~ H3*DIC
RCC ~ H4*DIQ

# Strategic capability pathway
SA  ~ H5*SP
RCC ~ H6*SA

# Main resilience pathway and environmental uncertainty
SCR ~ H7*RCC + EU_direct*EU

# Downstream outcomes
FSOC ~ H15*SCR
PSVC ~ H16*FSOC + H17*SCR

# Capability complementarity interactions
RCC ~ H12*RIxDIQ
RCC ~ H13*DIQxSA
RCC ~ H14*RIxSA

# Environmental uncertainty moderation
SCR ~ H11*RCCxEU

# Indirect effects
H8  := H1*H2
H9  := H3*H4
H10 := H5*H6

# Downstream indirect effects
H16_indirect := H15*H16
H17_indirect_via_FSOC := H15*H16
H_RCC_to_PSVC_via_SCR := H7*H17
H_RCC_to_PSVC_serial := H7*H15*H16
'
)

# 10. Estimate full interaction model with 5,000 bootstrap samples
set.seed(12345)

fit_interaction <- sem(
  model = full_interaction_model,
  data = interaction_data,
  estimator = "ML",
  std.lv = TRUE,
  missing = "fiml",
  se = "bootstrap",
  bootstrap = 5000
)

cat("\n\n================================================\n")
cat("RESULT 3: FULL STRUCTURAL MODEL WITH INTERACTIONS\n")
cat("================================================\n")

print(summary(
  fit_interaction,
  standardized = TRUE,
  rsquare = TRUE,
  ci = TRUE
))

# 11. Bootstrapped hypothesis results
cat("\n\n================================================\n")
cat("BOOTSTRAPPED HYPOTHESIS TESTING RESULTS\n")
cat("================================================\n")

boot_results <- parameterEstimates(
  fit_interaction,
  standardized = TRUE,
  ci = TRUE,
  boot.ci.type = "perc"
)

hypothesis_results <- boot_results %>%
  filter(
    (op == "~" & label %in% c(
      "H1", "H2", "H3", "H4", "H5", "H6", "H7",
      "H11", "H12", "H13", "H14", "H15", "H16", "H17"
    )) |
    (op == ":=" & label %in% c(
      "H8", "H9", "H10",
      "H16_indirect", "H17_indirect_via_FSOC",
      "H_RCC_to_PSVC_via_SCR", "H_RCC_to_PSVC_serial"
    ))
  ) %>%
  select(
    hypothesis = label,
    lhs, op, rhs,
    estimate, se, z, pvalue,
    ci.lower, ci.upper,
    std.all
  )

print(hypothesis_results, n = Inf)

# 12. R-squared values from full interaction model
cat("\n\n================================================\n")
cat("R-SQUARED VALUES: FULL INTERACTION MODEL\n")
cat("================================================\n")

print(inspect(fit_interaction, "r2"))

# 13. Compare R-squared values between the two structural models
cat("\n\n================================================\n")
cat("COMPARISON OF R-SQUARED VALUES\n")
cat("================================================\n")

r2_main <- inspect(fit_main, "r2")
r2_interaction <- inspect(fit_interaction, "r2")

r2_comparison <- data.frame(
  Construct = names(r2_main),
  R2_Main_Effects = as.numeric(r2_main),
  R2_Interactions = as.numeric(r2_interaction[names(r2_main)])
)

r2_comparison$Change_in_R2 <-
  r2_comparison$R2_Interactions - r2_comparison$R2_Main_Effects

print(r2_comparison)

# 14. Display model fit indices for reference
# Do NOT use interaction-model fit indices as the headline
# structural fit indices in the manuscript.

cat("\n\n================================================\n")
cat("INTERACTION MODEL FIT INDICES (REFERENCE ONLY)\n")
cat("================================================\n")

print(fitMeasures(
  fit_interaction,
  c("chisq", "df", "pvalue", "cfi", "tli",
    "rmsea", "rmsea.ci.lower", "rmsea.ci.upper", "srmr")
))

# 15. Save results to CSV files
write.csv(
  hypothesis_results,
  "Lavaan_Bootstrapped_Hypothesis_Results.csv",
  row.names = FALSE
)

write.csv(
  r2_comparison,
  "Lavaan_R2_Comparison.csv",
  row.names = FALSE
)

write.csv(
  standardizedSolution(fit_cfa),
  "Lavaan_Measurement_Model_Standardized_Results.csv",
  row.names = FALSE
)

write.csv(
  standardizedSolution(fit_main),
  "Lavaan_Main_Effects_Standardized_Results.csv",
  row.names = FALSE
)

write.csv(
  standardizedSolution(fit_interaction),
  "Lavaan_Interaction_Model_Standardized_Results.csv",
  row.names = FALSE
)

